# Solution 2 — PyTorch Feed-Forward Neural Network

A simple MLP trained on the same engineered features. With only ~730 rows an NN is overkill, but it's a useful reference: if a flexible non-linear learner doesn't clearly beat Ridge, that's evidence the data is close to linear-in-features.

Architecture: 2 hidden layers, ReLU, dropout, MSE loss, Adam.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

sns.set_theme(style='whitegrid')
torch.manual_seed(0); np.random.seed(0)

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['date'])
test  = pd.read_csv(DATA_DIR / 'test.csv',  parse_dates=['date'])
print(train.shape, test.shape)

## 1. Quick EDA

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0,0].plot(train['date'], train['sales'], lw=0.7); axes[0,0].set_title('Sales over time')
sns.histplot(train['sales'], kde=True, ax=axes[0,1]); axes[0,1].set_title('Sales distribution')
sns.boxplot(data=train, x='dow', y='sales', ax=axes[1,0]); axes[1,0].set_title('Sales by dow')
axes[1,1].scatter(train['price'], train['sales'], c=train['promo'], cmap='coolwarm', s=10, alpha=0.6)
axes[1,1].set_title('Sales vs price (colored by promo)'); axes[1,1].set_xlabel('price')
plt.tight_layout(); plt.show()

print('Weekly means:\n', train.groupby('dow')['sales'].mean().round(1))
print('Promo effect:\n', train.groupby('promo')['sales'].mean().round(1))

## 2. Features & scaling

In [ ]:
def make_features(df):
    X = pd.DataFrame(index=df.index)
    X['t'] = df['t']
    for d in range(1, 7):
        X[f'dow_{d}'] = (df['dow'] == d).astype(int)
    X['log_price'] = np.log(df['price'])
    is_weekend = df['dow'].isin([5,6]).astype(int)
    X['promo'] = df['promo']
    X['promo_x_weekend'] = df['promo'] * is_weekend
    angle = 2 * np.pi * df['t'] / 365.25
    X['yr_sin'] = np.sin(angle); X['yr_cos'] = np.cos(angle)
    return X

cutoff = len(train) - 90
tr, va = train.iloc[:cutoff], train.iloc[cutoff:]

X_tr, X_va = make_features(tr), make_features(va)
scaler_x = StandardScaler().fit(X_tr)
X_tr_s = scaler_x.transform(X_tr); X_va_s = scaler_x.transform(X_va)

# scale target for easier optimization
y_mean, y_std = tr['sales'].mean(), tr['sales'].std()
y_tr = (tr['sales'].values - y_mean) / y_std
y_va = (va['sales'].values - y_mean) / y_std

Xtr = torch.tensor(X_tr_s, dtype=torch.float32)
ytr = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)
Xva = torch.tensor(X_va_s, dtype=torch.float32)
yva = torch.tensor(y_va, dtype=torch.float32).unsqueeze(1)
Xtr.shape, Xva.shape

## 3. Model

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=32, p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x)

model = MLP(Xtr.shape[1])
opt = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)
loss_fn = nn.MSELoss()

train_losses, val_losses = [], []
for epoch in range(600):
    model.train(); opt.zero_grad()
    loss = loss_fn(model(Xtr), ytr); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        vl = loss_fn(model(Xva), yva).item()
    train_losses.append(loss.item()); val_losses.append(vl)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(train_losses, label='train'); ax.plot(val_losses, label='val')
ax.set_xlabel('epoch'); ax.set_ylabel('MSE (scaled)'); ax.legend()
ax.set_title('Training curves'); plt.show()

## 4. Validation performance

In [ ]:
model.eval()
with torch.no_grad():
    pred_va = model(Xva).numpy().ravel() * y_std + y_mean

rmse = np.sqrt(mean_squared_error(va['sales'], pred_va))
mape = mean_absolute_percentage_error(va['sales'], pred_va)
print(f'Validation  rmse={rmse:.2f}  mape={mape:.3f}')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(va['date'], va['sales'].values, label='actual', lw=1.2)
ax.plot(va['date'], pred_va, label='NN prediction', lw=1.2)
ax.legend(); ax.set_title('MLP on last 90 days'); plt.show()

## 5. Refit on all train, predict test

In [ ]:
X_all = make_features(train)
scaler_full = StandardScaler().fit(X_all)
X_all_s = scaler_full.transform(X_all)
X_te_s  = scaler_full.transform(make_features(test))
y_mean2, y_std2 = train['sales'].mean(), train['sales'].std()

Xall = torch.tensor(X_all_s, dtype=torch.float32)
yall = torch.tensor((train['sales'].values - y_mean2) / y_std2, dtype=torch.float32).unsqueeze(1)
Xte = torch.tensor(X_te_s, dtype=torch.float32)

final = MLP(Xall.shape[1])
opt = torch.optim.Adam(final.parameters(), lr=1e-2, weight_decay=1e-4)
for _ in range(600):
    final.train(); opt.zero_grad()
    loss_fn(final(Xall), yall).backward(); opt.step()

final.eval()
with torch.no_grad():
    pred_test = final(Xte).numpy().ravel() * y_std2 + y_mean2

out = pd.DataFrame({'date': test['date'], 'sales_pred': pred_test.round(2)})
out.to_csv('predictions_nn.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train['date'], train['sales'], lw=0.6, label='train')
ax.plot(test['date'], pred_test, lw=1.2, color='red', label='NN forecast')
ax.legend(); ax.set_title('MLP forecast for next 90 days'); plt.show()
out.head()